To load and plot a dataset of neural activity across population, in a PopAnal class object.


In [1]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:

# this is the path to the dataset
path = '/gorilla1/analyses/recordings/main/RSA/Diego-230615/agg_True-subtr_None-dist_euclidian_unbiased/SP_shape_loc/DFallpa.pkl'
path = "/lemur2/lucas/Dropbox/SCIENCE/FREIWALD_LAB/DATA/Xuan/DFallpa-Diego-230913-stroke-kilosort_if_exists-norm=None-combine=False-t1=-0.5-t2=2.1-quest=SP_BASE_stroke.pkl"

##### Load data

In [ ]:
DFallpa = pd.read_pickle(path)


In [ ]:
# This holds data across multiple brain regions (the "bregion" column)
# Ingore the "which_level", "event" and "twind" columns for now
display(DFallpa)


In [ ]:
# Pull out a single pa
pa = DFallpa["pa"].values[2]



In [ ]:
# This holds data for a single brain area
# The main data is here, in pa.X, an array with dimensions (channels, trials, timepoints)
pa.X.shape

# The labels for channels, trials, and times:
print("channels: ", pa.Chans)
print("trials: ", pa.Trials)
print("timepoints: ", pa.Times)




In [ ]:
# Each trials has associated "features", saved here, where each row of this dataframe matches a trial
pa.Xlabels["trials"]


In [ ]:
# e.g., to get the features for trial i, do this
trial = 10
features = pa.Xlabels["trials"].iloc[trial]
display(features)

### Debugging

In [2]:
    from neuralmonkey.scripts.analy_dfallpa_extract import extract_dfallpa_helper
    from neuralmonkey.classes.population_mult import load_handsaved_wrapper, dfpa_match_chans_across_pa_each_bregion
    from neuralmonkey.classes.population_mult import extract_single_pa
    from neuralmonkey.metadat.analy.anova_params import params_getter_euclidian_vars
    from neuralmonkey.classes.population_mult import dfpa_concatbregion_preprocess_clean_bad_channels, dfpa_concatbregion_preprocess_wrapper
    from pythonlib.tools.pandastools import append_col_with_grp_index
    import seaborn as sns
    import os
    from neuralmonkey.classes.population_mult import extract_single_pa
    from neuralmonkey.analyses.state_space_good import euclidian_distance_compute_trajectories_single, euclidian_distance_compute_trajectories
    from neuralmonkey.classes.population_mult import load_handsaved_wrapper, dfpa_concatbregion_preprocess_wrapper, dfpa_concat_bregion_to_combined_bregion
    from neuralmonkey.classes.population_mult import dfpa_concat_merge_pa_along_trials


In [3]:
### (1) load Grammar Dfallpa
animal = "Diego"
date  = 230913
version = "stroke"
combine = False
question = "RULE_ANBMCK_STROKE"


DFallpa = load_handsaved_wrapper(animal, date, version=version, combine_areas=combine, 
                                    question=question)
DFallpa = dfpa_concat_bregion_to_combined_bregion(DFallpa)

try:
    ### (2) Load SP data
    _question = "SP_BASE_stroke"
    _twind = [-0.5, 2.1]
    DFallpaSP = load_handsaved_wrapper(animal, date, version=version, combine_areas=combine, 
                                        question=_question, twind=_twind)
    DFallpaSP = dfpa_concat_bregion_to_combined_bregion(DFallpaSP)

    # Merge SP and grammar along chan indices
    DFallpa = dfpa_concat_merge_pa_along_trials(DFallpa, DFallpaSP)
    del DFallpaSP

except Exception as err:
    print(err)

# Make a copy of all PA before normalization
dfpa_concatbregion_preprocess_wrapper(DFallpa, animal, date)


Loading DFallpa from:  /lemur2/lucas/Dropbox/SCIENCE/FREIWALD_LAB/DATA/Xuan/DFallpa-Diego-230913-stroke-kilosort_if_exists-norm=None-combine=False-t1=-1.0-t2=1.8-quest=RULE_ANBMCK_STROKE.pkl
Loading DFallpa from:  /lemur2/lucas/Dropbox/SCIENCE/FREIWALD_LAB/DATA/Xuan/DFallpa-Diego-230913-stroke-kilosort_if_exists-norm=None-combine=False-t1=-0.5-t2=2.1-quest=SP_BASE_stroke.pkl
 
===  M1
chans (synt):  [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010, 1011, 1012, 1013, 1014, 1016, 1017, 1019, 1020, 1021, 1022, 1023, 1024, 1025, 1026, 1027, 1028]
chans (sp  ):  [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010, 1011, 1012, 1013, 1014, 1016, 1017, 1019, 1020, 1021, 1022, 1023, 1024, 1025, 1026, 1027, 1028]
chans (both):  [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010, 1011, 1012, 1013, 1014, 1016, 1017, 1019, 1020, 1021, 1022, 1023, 1024, 1025, 1026, 1027, 1028]
Fraction chans kept:  1.0 1.0
Making copy and replacing None with 'none', for:  behseq_shapes
* 

In [4]:
from neuralmonkey.scripts.analy_syntax_good_eucl_state import preprocess_dfallpa_motor_features
preprocess_dfallpa_motor_features(DFallpa)

### Sanity check

In [ ]:
path_split = "/lemur2/lucas/analyses/recordings/main/syntax_good/targeted_dim_redu_v2/run30/Diego-230913-q=RULE_ANBMCK_STROKE/bregion=preSMA/FITTING_subspc=('epoch', 'gridloc', 'DIFF_gridloc', 'chunk_rank', 'shape', 'rank_conj')-iter=0/pa_subspace.pkl"
path_full = "/lemur2/lucas/analyses/recordings/main/syntax_good/targeted_dim_redu_v2/run31/Diego-230913-q=RULE_ANBMCK_STROKE/bregion=preSMA/FITTING_subspc=('epoch', 'gridloc', 'DIFF_gridloc', 'chunk_rank', 'shape', 'rank_conj')-iter=0/pa_subspace.pkl"



In [8]:
import pickle

with open(path_split, "rb") as f:
    PAsplit = pickle.load(f)

with open(path_full, "rb") as f:
    PAfull = pickle.load(f)    

NameError: name 'dflab' is not defined

In [11]:
print(PAsplit.X.shape)
print(PAfull.X.shape)

(50, 2730, 1)
(50, 2730, 1)


In [18]:
for x, y in zip(sorted(PAfull.Trials), sorted(PAsplit.Trials)):
    print(x, y)

0 0
1 1
2 2
3 3
4 4
5 5
6 6
7 7
8 8
9 9
10 10
11 11
12 12
13 13
14 14
15 15
16 16
17 17
18 18
19 19
20 20
21 21
22 22
23 23
24 24
25 25
26 28
27 30
29 31
31 32
33 33
35 34
36 35
38 36
39 37
40 38
43 39
44 42
45 43
46 44
47 45
48 47
49 49
51 50
52 52
54 53
55 54
56 55
58 56
59 57
60 58
61 59
63 61
64 62
65 63
67 65
68 67
69 68
71 69
72 71
73 73
74 74
75 75
79 76
80 78
82 79
84 80
85 81
87 82
88 83
89 85
90 87
91 88
93 90
95 91
96 92
98 93
99 95
100 96
101 97
102 98
103 99
104 100
105 102
106 103
107 104
109 105
111 107
112 108
113 109
114 112
115 116
118 118
119 119
120 120
122 121
124 122
125 123
126 124
127 125
128 126
130 130
131 131
132 132
133 134
135 136
136 138
137 141
138 142
140 143
141 145
144 147
145 148
146 149
147 150
149 152
150 153
151 154
152 156
155 157
156 159
158 160
159 161
162 162
163 163
164 165
165 167
166 168
167 169
168 170
171 171
173 172
175 174
176 175
177 176
180 177
182 178
186 179
187 180
188 181
189 183
190 184
191 185
192 188
193 189
194 190
195 191
197 

In [ ]:
dflab = PAfull.Xlabels["trials"]

In [26]:
dflab["FEAT_num_strokes_task"]

0       7
1       7
2       7
3       7
4       7
       ..
2725    1
2726    1
2727    1
2728    1
2729    1
Name: FEAT_num_strokes_task, Length: 2730, dtype: int64

In [24]:
rang
# PAfull.slice_by_labels_filtdict({"task_kind":["prims_on_grid"]})
PAfull.slice_by_labels_filtdict({"FEAT_num_strokes_task":list(range(2, 20))})

pa.slice_by_labels_filtdict, using var=task_kind, n before filt: (50, 2730, 1)
pa.slice_by_labels_filtdict, using var=task_kind, n after filt: (50, 2616, 1)


In [31]:

# PAfull.slice_by_labels_filtdict({"task_kind":["prims_on_grid"]})
PAfull.slice_by_labels_filtdict({"FEAT_num_strokes_task":list(range(2, 20))})

pa.slice_by_labels_filtdict, using var=FEAT_num_strokes_task, n before filt: (50, 2730, 1)
pa.slice_by_labels_filtdict, using var=FEAT_num_strokes_task, n after filt: (50, 2616, 1)


##### PA has many helper functions to preprocess and plot the data. 

In [ ]:
# Example plotting

In [ ]:
chan = pa.Chans[4]

fig, ax = plt.subplots()
pa.plotwrapper_smoothed_fr_split_by_label("trials", "seqc_0_shape", ax, chan=chan)


fig, ax = plt.subplots()
pa.plotwrapper_smoothed_fr_split_by_label("trials", "seqc_0_shape", ax, chan=chan, plot_indiv=True)


In [ ]:
# Example processing. Here this picks out just a smaller time winodw


print("Times, before slicing: ", pa.Times)

twind = [-0.35, -0.3] # window: to only keep times within this window
pa = pa.slice_by_dim_values_wrapper("times", twind)

print("Times, after slicing:", pa.Times)